# 03_gcp_finetuner — Fine-tuning de BLIP en GCP

Notebook diseñado para correr en una VM de GCP con GPU (L4 o T4).

**Flujo recomendado:**
1. Ajustar la celda de configuración (especialmente `MAX_TRAIN_SAMPLES` y `BATCH_SIZE`).
2. Ejecutar el **smoke test** (`RUN_SMOKE_TEST = True`) para verificar que la GPU funciona con el stack completo.
3. Si el smoke test pasa, correr el **entrenamiento completo** (`RUN_FULL_TRAINING = True`).
4. Visualizar la curva de loss y correr el sanity check del checkpoint.

**Corrida preliminar:** `MAX_TRAIN_SAMPLES = 5000`, `MAX_VAL_SAMPLES = 500`, `MAX_TEST_SAMPLES = 300`  
**Corrida completa:** `MAX_TRAIN_SAMPLES = 15000`, `MAX_VAL_SAMPLES = 1500`, `MAX_TEST_SAMPLES = 1000`  
Poner `None` en cualquiera para usar el split completo.

**Target de entrenamiento:** `impression` (conclusión clínica corta, no `findings`).

## 0. Setup del entorno

In [ ]:
# Descomentar y ejecutar solo la primera vez en una VM nueva.
# !pip install -r requirements.txt

In [ ]:
from pathlib import Path
import os
import sys
import torch

# Detectar raíz del repo de forma robusta (cwd puede ser notebooks/ o la raíz)
CWD = Path.cwd()
if (CWD / "src").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "src").exists():
    REPO_ROOT = CWD.parent
else:
    raise RuntimeError(
        "No pude detectar la raíz del repo. "
        "Ejecutá este notebook desde la raíz del proyecto o desde notebooks/."
    )

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repo root:", REPO_ROOT)
print("Python:", sys.executable)
print("Torch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM total:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("Sin CUDA — usar solo smoke tests en esta máquina.")

## 1. Configuración central

**Cambiar aquí** antes de cada corrida. Todo el resto del notebook usa estas variables.

In [ ]:
# ── Subsets ───────────────────────────────────────────────────────────────────
# None = usar el split completo. Los primeros N índices (ya mezclados con seed=42).
#
# Corrida preliminar : train=5000  val=500   test=300
# Corrida completa   : train=15000 val=1500  test=1000
# Todo               : None        None      None
MAX_TRAIN_SAMPLES = 5000
MAX_VAL_SAMPLES   = 500
MAX_TEST_SAMPLES  = 300

# ── Modelo ────────────────────────────────────────────────────────────────────
MODEL_DIR  = Path("models/blip_base")   # carga desde disco si existe, si no baja de HF
CACHE_DIR  = Path("data/hf_cache")

# OUTPUT_DIR se deriva automáticamente para que corridas distintas no se pisen.
# 5k → models/blip_finetuned_5k/  |  15k → models/blip_finetuned_15k/  |  None → models/blip_finetuned_full/
_n_tag    = f"{MAX_TRAIN_SAMPLES // 1000}k" if MAX_TRAIN_SAMPLES else "full"
OUTPUT_DIR = Path(f"models/blip_finetuned_{_n_tag}")

# ── Splits ────────────────────────────────────────────────────────────────────
TRAIN_INDICES    = Path("data/splits/train_indices.json")
VAL_INDICES      = Path("data/splits/val_indices.json")
TEST_INDICES     = Path("data/splits/test_indices.json")
SELECTED_INDICES = Path("data/selected_indices.json")

# ── Hiperparámetros ───────────────────────────────────────────────────────────
EPOCHS           = 3
BATCH_SIZE       = 8      # batch real por step (ajustar según VRAM disponible)
GRAD_ACCUM_STEPS = 1      # batch efectivo = BATCH_SIZE * GRAD_ACCUM_STEPS
                          # con L4/T4 y batch=8 no hace falta acumulación
NUM_WORKERS      = 2
MAX_LENGTH       = 128    # longitud máxima de tokenización del texto target
ENCODER_LR       = 5e-6
DECODER_LR       = 1e-5
WEIGHT_DECAY     = 0.01
WARMUP_RATIO     = 0.05
PATIENCE         = 2      # early stopping: épocas sin mejora antes de cortar
SEED             = 42
TEXT_COL         = "impression"  # no cambiar — ver sección 8.5 del CLAUDE.md

# ── Flags de ejecución ────────────────────────────────────────────────────────
RUN_SMOKE_TEST    = False  # poner True para verificar el stack antes del entrenamiento completo
RUN_FULL_TRAINING = False  # poner True para lanzar el entrenamiento completo

# ── Resumen de configuración ──────────────────────────────────────────────────
effective_batch = BATCH_SIZE * GRAD_ACCUM_STEPS
print("=" * 60)
print(f"MAX_TRAIN_SAMPLES : {MAX_TRAIN_SAMPLES}")
print(f"MAX_VAL_SAMPLES   : {MAX_VAL_SAMPLES}")
print(f"MAX_TEST_SAMPLES  : {MAX_TEST_SAMPLES}")
print(f"EPOCHS            : {EPOCHS}")
print(f"BATCH_SIZE        : {BATCH_SIZE}  (efectivo: {effective_batch})")
print(f"GRAD_ACCUM_STEPS  : {GRAD_ACCUM_STEPS}")
print(f"ENCODER_LR        : {ENCODER_LR}")
print(f"DECODER_LR        : {DECODER_LR}")
print(f"PATIENCE          : {PATIENCE}")
print(f"TEXT_COL          : {TEXT_COL}")
print(f"OUTPUT_DIR        : {OUTPUT_DIR}")
print("=" * 60)

## 2. Verificación de paths y splits

In [ ]:
import json

# Verificar archivos requeridos
required = [TRAIN_INDICES, VAL_INDICES, TEST_INDICES, SELECTED_INDICES]
missing = [p for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(f"Faltan archivos requeridos: {missing}")

# Cargar índices completos
with open(TRAIN_INDICES) as f:
    all_train_indices = json.load(f)
with open(VAL_INDICES) as f:
    all_val_indices = json.load(f)
with open(TEST_INDICES) as f:
    all_test_indices = json.load(f)
with open(SELECTED_INDICES) as f:
    selected_indices = json.load(f)

# Aplicar subsets (los índices ya están mezclados con seed=42, se toman los primeros N)
def _apply_limit(indices, limit, name):
    if limit is not None:
        subset = indices[:limit]
        print(f"{name}: {len(subset)} / {len(indices)} índices")
        return subset
    print(f"{name}: {len(indices)} índices (completo)")
    return indices

train_indices = _apply_limit(all_train_indices, MAX_TRAIN_SAMPLES, "Train")
val_indices   = _apply_limit(all_val_indices,   MAX_VAL_SAMPLES,   "Val  ")
test_indices  = _apply_limit(all_test_indices,  MAX_TEST_SAMPLES,  "Test ")
print(f"Selected       : {len(selected_indices)} índices (fijo, no se modifica)")

# Verificar overlaps
selected_set = set(selected_indices)
train_set    = set(train_indices)
val_set      = set(val_indices)
test_set     = set(test_indices)

checks = [
    ("train ∩ val == 0",      len(train_set & val_set) == 0),
    ("train ∩ test == 0",     len(train_set & test_set) == 0),
    ("val ∩ test == 0",       len(val_set & test_set) == 0),
    ("selected ∩ train == 0", len(selected_set & train_set) == 0),
    ("selected ∩ val == 0",   len(selected_set & val_set) == 0),
]

print()
for label, ok in checks:
    print(f"{'OK' if ok else 'PROBLEMA'}: {label}")

if not all(ok for _, ok in checks):
    raise RuntimeError("Fallo en verificación de splits — revisar antes de continuar.")

print("\nSplits verificados correctamente.")

## 3. Carga de modelo y dataset

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

from src.models.blip_loader import load_model_and_processor
from src.data.utils import load_mimic_dataset
from src.models.finetuner import set_seed

set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

print("Cargando modelo...")
model, processor = load_model_and_processor(model_dir=MODEL_DIR, device=device)
print("Modelo cargado.")

print("Cargando dataset...")
ds = load_mimic_dataset(cache_dir=str(CACHE_DIR))
hf_split = ds["train"]
print(f"Dataset cargado: {len(hf_split)} filas totales.")

## 4. Creación de DataLoaders

In [ ]:
from src.data.dataloader import create_dataloader

train_loader = create_dataloader(
    hf_split=hf_split,
    processor=processor,
    indices=train_indices,
    text_col=TEXT_COL,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    max_length=MAX_LENGTH,
)

val_loader = create_dataloader(
    hf_split=hf_split,
    processor=processor,
    indices=val_indices,
    text_col=TEXT_COL,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    max_length=MAX_LENGTH,
)

print(f"Train DataLoader: {len(train_loader)} batches (batch_size={BATCH_SIZE})")
print(f"Val   DataLoader: {len(val_loader)} batches")
print(f"Batch efectivo  : {BATCH_SIZE * GRAD_ACCUM_STEPS}")

## 5. Smoke test en GPU

Corre 2 batches de train y 1 de val para verificar que el stack funciona sin errores.  
Poner `RUN_SMOKE_TEST = True` en la celda de configuración para ejecutarlo.

In [ ]:
from src.models.finetuner import run_finetuning
from src.models.blip_loader import load_model_and_processor

if not RUN_SMOKE_TEST:
    print("RUN_SMOKE_TEST=False — saltando.")
else:
    if not torch.cuda.is_available():
        raise RuntimeError("Smoke test requiere CUDA.")

    smoke_output = Path("models/blip_finetuned_smoke")

    # Reload fresco para no contaminar el estado del modelo antes del entrenamiento real
    smoke_model, smoke_processor = load_model_and_processor(model_dir=MODEL_DIR, device=device)

    smoke_history = run_finetuning(
        model=smoke_model,
        processor=smoke_processor,
        train_dataloader=train_loader,
        val_dataloader=val_loader,
        num_epochs=1,
        output_dir=smoke_output,
        device=device,
        patience=1,
        encoder_lr=ENCODER_LR,
        decoder_lr=DECODER_LR,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        grad_accum_steps=GRAD_ACCUM_STEPS,
        use_amp=True,
        max_train_batches=2,
        max_val_batches=1,
        skip_checkpoint_save=True,
    )

    del smoke_model
    torch.cuda.empty_cache()

    r = smoke_history[0]
    print("Smoke test OK.")
    print(f"  train_loss={r['train_loss']:.4f}  val_loss={r['val_loss']:.4f}  "
          f"perplexity={r['val_perplexity']:.1f}  token_acc={r['val_token_acc']:.3f}")
    print("Podés poner RUN_FULL_TRAINING=True para continuar.")

## 6. Fine-tuning completo

Guarda un checkpoint por época en `OUTPUT_DIR/epoch_N/` y el mejor en `OUTPUT_DIR/best/`.  
El historial de losses se guarda en `OUTPUT_DIR/training_history.json` al final de cada época.

Poner `RUN_FULL_TRAINING = True` en la celda de configuración para ejecutarlo.

In [ ]:
if not RUN_FULL_TRAINING:
    print("RUN_FULL_TRAINING=False — saltando.")
else:
    if not torch.cuda.is_available():
        raise RuntimeError("El entrenamiento completo requiere CUDA.")

    history = run_finetuning(
        model=model,
        processor=processor,
        train_dataloader=train_loader,
        val_dataloader=val_loader,
        num_epochs=EPOCHS,
        output_dir=OUTPUT_DIR,
        device=device,
        patience=PATIENCE,
        encoder_lr=ENCODER_LR,
        decoder_lr=DECODER_LR,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        grad_accum_steps=GRAD_ACCUM_STEPS,
        use_amp=True,
    )

    print("\nFine-tuning finalizado.")
    print(f"Épocas entrenadas: {len(history)}")
    best_epoch = min(history, key=lambda r: r["val_loss"])
    print(f"Mejor época      : {int(best_epoch['epoch'])}")
    print(f"  val_loss       : {best_epoch['val_loss']:.4f}")
    print(f"  val_perplexity : {best_epoch['val_perplexity']:.1f}")
    print(f"  val_token_acc  : {best_epoch['val_token_acc']:.3f}")

## 7. Curva de loss

In [ ]:
import matplotlib.pyplot as plt

history_path = OUTPUT_DIR / "training_history.json"

if not history_path.exists():
    print("No hay training_history.json todavía. Ejecutar primero el entrenamiento.")
else:
    with open(history_path) as f:
        history_data = json.load(f)

    print(f"Épocas registradas: {len(history_data)}")
    print(f"{'Época':>6}  {'train_loss':>10}  {'val_loss':>9}  {'perplexity':>10}  {'token_acc':>9}")
    print("-" * 55)
    for row in history_data:
        print(f"{int(row['epoch']):>6}  {row['train_loss']:>10.4f}  {row['val_loss']:>9.4f}"
              f"  {row['val_perplexity']:>10.1f}  {row['val_token_acc']:>9.3f}")

    epochs_x     = [r["epoch"] for r in history_data]
    train_losses = [r["train_loss"] for r in history_data]
    val_losses   = [r["val_loss"] for r in history_data]
    perplexities = [r["val_perplexity"] for r in history_data]
    token_accs   = [r["val_token_acc"] for r in history_data]

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    n = MAX_TRAIN_SAMPLES if MAX_TRAIN_SAMPLES else "full"

    axes[0].plot(epochs_x, train_losses, marker="o", label="train")
    axes[0].plot(epochs_x, val_losses,   marker="o", label="val")
    axes[0].set_title("Loss (cross-entropy)")
    axes[0].set_xlabel("Época")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs_x, perplexities, marker="o", color="tab:orange")
    axes[1].set_title("Val Perplexity")
    axes[1].set_xlabel("Época")
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(epochs_x, token_accs, marker="o", color="tab:green")
    axes[2].set_title("Val Token Accuracy")
    axes[2].set_xlabel("Época")
    axes[2].set_ylim(0, 1)
    axes[2].grid(True, alpha=0.3)

    fig.suptitle(f"Fine-tuning BLIP — n_train={n}, batch_eff={BATCH_SIZE * GRAD_ACCUM_STEPS}")
    fig.tight_layout()

    # el tag de corrida va en el nombre del archivo para no pisar entre corridas
    plot_path = Path(f"outputs/finetuning/loss_curve_{_n_tag}.png")
    plot_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(plot_path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Figura guardada en:", plot_path)

## 8. Sanity check del checkpoint fine-tuneado

Carga `OUTPUT_DIR/best/` y genera captions sobre las primeras 3 imágenes de `selected_indices.json`.  
Muestra imagen, caption generado y referencia `impression`.

In [ ]:
best_dir = OUTPUT_DIR / "best"

if not best_dir.exists():
    print(f"{best_dir} no existe todavía. Ejecutar primero el entrenamiento completo.")
else:
    ft_model, ft_processor = load_model_and_processor(model_dir=best_dir, device=device)
    ft_model.eval()

    n_show = min(3, len(selected_indices))
    fig, axes = plt.subplots(1, n_show, figsize=(5 * n_show, 5))
    if n_show == 1:
        axes = [axes]

    for ax, idx in zip(axes, selected_indices[:n_show]):
        sample = hf_split[idx]
        image  = sample["image"].convert("RGB")
        ref    = sample.get(TEXT_COL, "")

        inputs = ft_processor(images=image, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            out_ids = ft_model.generate(**inputs, max_new_tokens=40, num_beams=1)
        caption = ft_processor.decode(out_ids[0], skip_special_tokens=True)

        ax.imshow(image, cmap="gray")
        ax.axis("off")
        ax.set_title(f"idx={idx}", fontsize=9)

        print(f"\n--- idx={idx} ---")
        print(f"  Caption FT : {caption}")
        print(f"  Referencia : {ref[:200]}")

    fig.tight_layout()
    plt.show()

    if torch.cuda.is_available():
        del ft_model
        torch.cuda.empty_cache()

## 9. Resumen de checkpoints guardados

In [ ]:
if not OUTPUT_DIR.exists():
    print("OUTPUT_DIR no existe todavía.")
else:
    print(f"Contenido de {OUTPUT_DIR}:")
    for p in sorted(OUTPUT_DIR.iterdir()):
        if p.is_dir():
            size_mb = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e6
            print(f"  {p.name}/  ({size_mb:.0f} MB)")
        else:
            size_kb = p.stat().st_size / 1e3
            print(f"  {p.name}  ({size_kb:.1f} KB)")

## 10. Generación de captions sobre el test set

Genera captions con el modelo fine-tuneado sobre todos los `test_indices` y los guarda en:

```
outputs/captions/captions_ft_{n_tag}.json
```

Formato por entrada: `{"idx": int, "generated": str, "reference": str}`.

Este archivo es la entrada directa al notebook 05 para calcular BLEU/CIDEr/METEOR.  
También se genera `captions_base_{n_tag}.json` con el modelo base para comparar.

In [ ]:
def generate_captions(model, processor, hf_split, indices, device, batch_size=16, max_new_tokens=40):
    """Genera captions sobre una lista de índices. Devuelve lista de dicts."""
    model.eval()
    results = []

    for i in range(0, len(indices), batch_size):
        batch_indices = indices[i : i + batch_size]
        images, refs = [], []

        for idx in batch_indices:
            sample = hf_split[idx]
            images.append(sample["image"].convert("RGB"))
            refs.append(sample.get(TEXT_COL, "") or "")

        inputs = processor(images=images, return_tensors="pt", padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            out_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=1)

        for idx, ref, ids in zip(batch_indices, refs, out_ids):
            caption = processor.decode(ids, skip_special_tokens=True)
            results.append({"idx": int(idx), "generated": caption, "reference": ref})

        if (i // batch_size + 1) % 10 == 0:
            print(f"  {len(results)}/{len(indices)} imágenes procesadas...")

    return results


captions_dir = Path("outputs/captions")
captions_dir.mkdir(parents=True, exist_ok=True)

best_dir = OUTPUT_DIR / "best"

if not best_dir.exists():
    print(f"{best_dir} no existe — ejecutar primero el entrenamiento completo.")
else:
    # ── Captions del modelo fine-tuneado ─────────────────────────────────────
    print(f"Generando captions FT sobre {len(test_indices)} imágenes de test...")
    ft_model, ft_processor = load_model_and_processor(model_dir=best_dir, device=device)

    captions_ft = generate_captions(ft_model, ft_processor, hf_split, test_indices, device)

    ft_path = captions_dir / f"captions_ft_{_n_tag}.json"
    with open(ft_path, "w", encoding="utf-8") as f:
        json.dump(captions_ft, f, indent=2, ensure_ascii=False)
    print(f"Guardado: {ft_path}  ({len(captions_ft)} entradas)")

    del ft_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ── Captions del modelo base (para comparar) ──────────────────────────────
    print(f"\nGenerando captions BASE sobre las mismas {len(test_indices)} imágenes...")
    base_model, base_processor = load_model_and_processor(model_dir=MODEL_DIR, device=device)

    captions_base = generate_captions(base_model, base_processor, hf_split, test_indices, device)

    base_path = captions_dir / f"captions_base_{_n_tag}.json"
    with open(base_path, "w", encoding="utf-8") as f:
        json.dump(captions_base, f, indent=2, ensure_ascii=False)
    print(f"Guardado: {base_path}  ({len(captions_base)} entradas)")

    del base_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ── Preview de 3 ejemplos ─────────────────────────────────────────────────
    print("\nEjemplos (primeros 3):")
    for entry_ft, entry_base in zip(captions_ft[:3], captions_base[:3]):
        print(f"\n  idx={entry_ft['idx']}")
        print(f"  BASE : {entry_base['generated']}")
        print(f"  FT   : {entry_ft['generated']}")
        print(f"  REF  : {entry_ft['reference'][:120]}")